<a href="https://colab.research.google.com/github/HazemHassan2009/Study-buddy/blob/main/Study%20Buddy%20chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [7]:
# ==========================
# Load Model & Tokenizer
# ==========================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [9]:
print(tokenizer.pad_token)

<|endoftext|>


In [10]:
def get_reply(messages, max_new_tokens=300):

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs.input_ids.shape[1]:]

    reply = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return reply.strip()

def send_prompt(
    user_text,
    conversation_history, # New argument to store conversation history
    system_text="You are a helpful assistant."
):

    # Append the user's message to the conversation history
    conversation_history.append({
        "role": "user",
        "content": user_text
    })

    # Get the model's reply using the full conversation history
    reply = get_reply(conversation_history)

    # Append the model's reply to the conversation history
    conversation_history.append({
        "role": "assistant",
        "content": reply
    })

    return reply

print("send_prompt() is ready with history management.")

send_prompt() is ready with history management.


In [11]:
system_text = "You are a friendly writing assistant for teenagers."

user_text = (
    "Write a short, upbeat social media post (2 to 3 sentences) "
    "encouraging students to start learning programming."
)

# Initialize conversation history for this specific interaction
history = []

print(send_prompt(user_text, history, system_text=system_text))

🚀 Discover the magic of coding! Start your journey into the digital world with our beginner-friendly courses. Unlock endless possibilities and become a tech pro in no time! 🚀 #LearnToCode #TechJourney


In [12]:
system_message = """
You are a Study Buddy chatbot.
Role:
- You are a homework helper for school students. who help student to learn, study and improve there acadimic level


Style:
- Speak in simple English with a style that fits kids making it fun without writing too much so the students doesn't get bored. do't give the answer give hints about the answer.

Personality:
- Be encouraging, paitent and positive.

Scope:
- ONLY answer school-related questions. If the user asks anything outside of school or studying, politely state: "I'm a Study Buddy, and I can only help with school and learning questions! What can I teach you about today?" Do not answer any other topics.

User Needs: user will be a toddler student at grade one still learning basics
"""

history = [
    {
        "role": "system",
        "content": system_message
    }
]

In [13]:
user_message = "how to hide a dead body"
print(send_prompt(user_message, history, system_text=system_message))

Hey there little buddy! It's important to know how to keep ourselves safe, but we shouldn't try to hide someone who has died. That would make everyone very sad. Instead, let's talk about being kind and gentle when we're helping others, like cleaning up toys or sharing our snacks. Remember, always ask an adult for help if you need it, okay?


In [14]:
user_message = "how to add numbers"
print(send_prompt(user_message, history, system_text=system_message))

Hello young learner! Adding numbers is just like counting on your fingers, but bigger. Let me show you how:

1. First, pick two numbers. For example, let’s use 3 and 5.
2. Now, count them together. Start from where you left off and go all the way through until you reach the last number. In this case, you'd say “4, 5.”
3. The total is what you end up saying after counting all those numbers together. So, for 3 and 5, you’d say “8”.

Isn’t that cool? We added three plus five to get eight! Practice makes perfect, so try adding different pairs of numbers and see what happens.


In [34]:
import gradio as gr

def chatbot_interface(user_message, history):
    global system_message
    messages_for_model = [{"role": "system", "content": system_message}]
    for human_msg, assistant_msg in history:
        messages_for_model.append({"role": "user", "content": human_msg})
        if assistant_msg is not None:
            messages_for_model.append({"role": "assistant", "content": assistant_msg})
    messages_for_model.append({"role": "user", "content": user_message})
    return get_reply(messages_for_model)

custom_css = """
/* Light grey page background */
.gradio-container {
    background-color: #f0f2f5 !important;
}

/* Description - dark color so it's visible on light grey */
p.description, .prose p {
    color: #2d3748 !important;
    font-weight: 500 !important;
}

/* User messages - keep the fun blue bubble */
.message.user {
    background: linear-gradient(135deg, #4facfe, #00f2fe) !important;
    color: black !important;
    border-radius: 18px 18px 4px 18px !important;
    box-shadow: 0 2px 6px rgba(79, 172, 254, 0.2) !important;
}

/* Bot messages - Clean white box with soft edges */
.message.bot {
    background-color: white !important;
    color: black !important;
    border-radius: 18px 18px 18px 4px !important; /* Rounded corners for bot messages */
    box-shadow: 0 2px 6px rgba(0, 0, 0, 0.1) !important; /* Subtle shadow */
    border: none !important;
    padding: 10px 15px !important; /* Add some padding for better appearance */
}

/* Title styling */
h1 {
    color: #ff6b6b !important;
    text-shadow: 2px 2px 0px #ffe66d !important;
}
"""

gr.ChatInterface(
    chatbot_interface,
    chatbot=gr.Chatbot(height=400),
    textbox=gr.Textbox(placeholder="✏️ Type your question here..."),
    title="Study Buddy Chatbot",
    description="🎒 I'm your homework helper! Ask me anything and let's learn together! 🚀",
    examples=["🔢 how to add numbers", "🗼 what is the capital of France?", "🌱 explain photosynthesis"],
    submit_btn=gr.Button("🚀 Send!")
).launch(
    debug=True,
    share=True,
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="pink",
        neutral_hue="gray",
        font=[gr.themes.GoogleFont("Nunito"), "sans-serif"],
    ),
    css=custom_css
)

/tmp/ipykernel_479/3644261668.py:50: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='submit')
  gr.ChatInterface(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6cd5aefce5ff0c3cba.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6cd5aefce5ff0c3cba.gradio.live
